# Phone Dataset Complete Analysis

This notebook contains a complete data analysis pipeline for the flipkart_phones_enriched.csv dataset.

Pipeline:
1. Null Audit
2. Visualize Missing Values
3. Missing Mechanism Detection (MCAR/MAR/MNAR)
4. Imputation
5. Confirm Zero Nulls
6. Post-Imputation Distribution Check
7. Full EDA
8. Feature Engineering
9. Feature Selection
10. Build Regression Models
11. Clustering & Segmentation
12. Streamlit App Generation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from scipy import stats
from scipy.stats import pointbiserialr, ks_2samp
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')
import joblib
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")
print("All imports successful!")

In [ ]:
df = pd.read_csv('flipkart_phones_enriched.csv')
print(f"Dataset Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
# STEP 1 — NULL AUDIT
print("="*60)
print("STEP 1: NULL AUDIT")
print("="*60)

null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({'Null Count': null_counts, 'Null %': null_pct})
null_df = null_df[null_df['Null Count'] > 0].sort_values('Null Count', ascending=False)

if len(null_df) > 0:
    print(f"\nColumns with null values ({len(null_df)} of {len(df.columns)}):")
    print(null_df.to_string())
else:
    print("\nNo null values found in any column!")
print(f"\nTotal rows: {len(df)}")
print(f"Total null cells: {df.isnull().sum().sum()}")

In [ ]:
# STEP 2 — VISUALIZE MISSING VALUES
print("="*60)
print("STEP 2: VISUALIZE MISSING VALUES")
print("="*60)

if null_df.shape[0] > 0:
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    msno.matrix(df, ax=axes[0], sparkline=False, fontsize=10)
    axes[0].set_title('Missing Value Matrix', fontsize=14, fontweight='bold')
    msno.heatmap(df, ax=axes[1], fontsize=10)
    axes[1].set_title('Missing Value Heatmap (Correlation)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('missing_values_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("No missing values to visualize!")

In [ ]:
# STEP 3 — MISSING MECHANISM DETECTION
print("="*60)
print("STEP 3: MISSING MECHANISM DETECTION")
print("="*60)

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
null_columns = null_df.index.tolist()

mechanism_results = {}

# Create null indicator columns
for col in null_columns:
    df[f'{col}_is_null'] = df[col].isnull().astype(int)

null_indicator_cols = [f'{col}_is_null' for col in null_columns]

# --- MCAR Detection: Point-biserial correlation ---
print("\n--- MCAR Detection (Point-Biserial Correlation) ---")
mcar_candidates = []
for null_col in null_columns:
    null_ind = df[f'{null_col}_is_null']
    other_numeric = [c for c in numeric_cols if c != null_col]
    
    correlations = []
    for num_col in other_numeric:
        valid = df[[num_col]].join(null_ind).dropna()
        if valid[num_col].nunique() > 1 and null_ind.sum() > 0:
            corr, pval = pointbiserialr(valid[f'{null_col}_is_null'], valid[num_col])
            correlations.append({'Feature': num_col, 'Correlation': corr, 'P-Value': pval})
    
    corr_df = pd.DataFrame(correlations)
    if len(corr_df) > 0:
        max_corr = corr_df['Correlation'].abs().max()
        print(f"\n{null_col}:")
        print(f"  Max |correlation| with other features: {max_corr:.4f}")
        if max_corr < 0.1:
            print(f"  -> Classified as MCAR (no significant correlation)")
            mcar_candidates.append(null_col)
        else:
            sig_corr = corr_df[corr_df['P-Value'] < 0.05]
            if len(sig_corr) > 0:
                print(f"  Significant correlations:")
                for _, row in sig_corr.iterrows():
                    print(f"    {row['Feature']}: corr={row['Correlation']:.4f}, p={row['P-Value']:.4f}")

# --- MAR Detection ---
print("\n\n--- MAR Detection (Null Correlation Heatmap) ---")
if len(null_indicator_cols) > 1:
    null_corr = df[null_indicator_cols].corr()
    print("Null-indicator correlation matrix:")
    print(null_corr.round(3))
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(null_corr, annot=True, cmap='RdYlBu_r', center=0, fmt='.3f')
    plt.title('Null-Indicator Correlation Heatmap (MAR Detection)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('mar_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

# Check if nulls in one column correlate with values in another
mar_candidates = []
for null_col in null_columns:
    if null_col not in mcar_candidates:
        null_ind = df[f'{null_col}_is_null']
        for num_col in numeric_cols:
            if num_col != null_col:
                valid = df[[num_col]].join(null_ind).dropna()
                if len(valid) > 0 and valid[num_col].nunique() > 1:
                    corr, pval = pointbiserialr(valid[f'{null_col}_is_null'], valid[num_col])
                    if abs(corr) >= 0.1 and pval < 0.05:
                        print(f"\nMAR evidence: {null_col} nulls correlate with {num_col} (corr={corr:.4f}, p={pval:.4f})")
                        if null_col not in mar_candidates:
                            mar_candidates.append(null_col)

# --- MNAR Detection ---
print("\n\n--- MNAR Detection (KS Test on Grouped Distributions) ---")
mnar_candidates = []
for null_col in null_columns:
    if null_col not in mcar_candidates and null_col not in mar_candidates:
        null_ind = df[f'{null_col}_is_null']
        other_numeric = [c for c in numeric_cols if c != null_col]
        
        print(f"\n{null_col}:")
        for num_col in other_numeric[:3]:
            null_group = df.loc[null_ind == 1, num_col].dropna()
            notnull_group = df.loc[null_ind == 0, num_col].dropna()
            
            if len(null_group) > 0 and len(notnull_group) > 0:
                ks_stat, ks_p = ks_2samp(null_group, notnull_group)
                print(f"  {num_col}: KS stat={ks_stat:.4f}, p={ks_p:.4f}", end="")
                if ks_p < 0.05:
                    print(f" -> SIGNIFICANT (potential MNAR)")
                else:
                    print()
        
        # If not classified as MCAR or MAR, treat as MNAR
        if null_col not in mar_candidates:
            mnar_candidates.append(null_col)

# --- Summary Table ---
print("\n\n" + "="*60)
print("MISSING MECHANISM SUMMARY")
print("="*60)
summary = []
for col in null_columns:
    if col in mcar_candidates:
        mech = 'MCAR'
    elif col in mar_candidates:
        mech = 'MAR'
    elif col in mnar_candidates:
        mech = 'MNAR'
    else:
        mech = 'Unclassified'
    summary.append({'Column': col, 'Mechanism': mech, 'Null Count': df[col].isnull().sum()})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

# Store for later use
null_mechanisms = {row['Column']: row['Mechanism'] for _, row in summary_df.iterrows()}

In [ ]:
# STEP 4 — IMPUTATION
print("="*60)
print("STEP 4: IMPUTATION")
print("="*60)

df_imputed = df.copy()
imputed_columns = []

for col, mech in null_mechanisms.items():
    print(f"\nImputing {col} ({mech}):")
    
    if mech == 'MCAR':
        # Median imputation
        median_val = df_imputed[col].median()
        df_imputed[col].fillna(median_val, inplace=True)
        print(f"  Median imputation with value: {median_val}")
        imputed_columns.append(col)
    
    elif mech == 'MAR':
        # KNN Imputer with k=5 using all numeric features
        knn_features = [c for c in numeric_cols if c != col]
        knn_data = df_imputed[knn_features + [col]].copy()
        imputer = KNNImputer(n_neighbors=5)
        imputed_vals = imputer.fit_transform(knn_data)
        df_imputed[col] = imputed_vals[:, -1]
        print(f"  KNN Imputation (k=5) using features: {knn_features[:5]}...")
        imputed_columns.append(col)
    
    elif mech == 'MNAR':
        # Add binary flag then median impute
        flag_name = f'{col}_was_missing'
        df_imputed[flag_name] = df_imputed[col].isnull().astype(int)
        median_val = df_imputed[col].median()
        df_imputed[col].fillna(median_val, inplace=True)
        print(f"  Added flag column: {flag_name}")
        print(f"  Then median imputation with value: {median_val}")
        imputed_columns.append(col)
        imputed_columns.append(flag_name)

# Drop null indicator columns
df_imputed.drop(columns=[f'{c}_is_null' for c in null_columns], inplace=True, errors='ignore')

print(f"\nTotal columns imputed: {len(imputed_columns)}")

In [ ]:
# STEP 5 — CONFIRM ZERO NULLS
print("="*60)
print("STEP 5: CONFIRM ZERO NULLS")
print("="*60)

remaining_nulls = df_imputed.isnull().sum()
cols_with_nulls = remaining_nulls[remaining_nulls > 0]

if len(cols_with_nulls) == 0:
    print("\nSUCCESS: All columns show 0 nulls!")
else:
    print("\nWARNING: Some columns still have nulls:")
    print(cols_with_nulls)

print(f"\nTotal null cells remaining: {df_imputed.isnull().sum().sum()}")
print(f"Dataset shape: {df_imputed.shape}")

In [ ]:
# STEP 6 — POST-IMPUTATION DISTRIBUTION CHECK
print("="*60)
print("STEP 6: POST-IMPUTATION DISTRIBUTION CHECK")
print("="*60)

# Columns that were imputed (exclude flag columns)
original_imputed = [c for c in imputed_columns if '_was_missing' not in c]

for col in original_imputed:
    print(f"\n--- {col} ---")
    
    # Get original non-null values
    orig_vals = df[col].dropna()
    post_vals = df_imputed[col].dropna()
    
    # Stats table
    stats_data = {
        'Metric': ['Mean', 'Std', 'Skew'],
        'Before': [orig_vals.mean(), orig_vals.std(), orig_vals.skew()],
        'After': [post_vals.mean(), post_vals.std(), post_vals.skew()]
    }
    stats_table = pd.DataFrame(stats_data).round(4)
    print(stats_table.to_string(index=False))
    
    # KS test
    ks_stat, ks_p = ks_2samp(orig_vals, post_vals)
    if ks_p < 0.05:
        print(f"WARNING: Distribution shift detected (KS p={ks_p:.4f} < 0.05)")
    else:
        print(f"OK: No significant distribution shift (KS p={ks_p:.4f})")
    
    # Plot
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Histogram overlay
    axes[0].hist(orig_vals, bins=30, alpha=0.6, label='Before Imputation', density=True, color='blue')
    axes[0].hist(post_vals, bins=30, alpha=0.6, label='After Imputation', density=True, color='red')
    axes[0].set_title(f'{col}: Histogram Overlay', fontweight='bold')
    axes[0].legend()
    axes[0].set_xlabel(col)
    
    # 2. Boxplot
    axes[1].boxplot([orig_vals.values, post_vals.values], labels=['Before', 'After'])
    axes[1].set_title(f'{col}: Boxplot Comparison', fontweight='bold')
    axes[1].set_ylabel(col)
    
    # 3. KDE plot
    orig_vals.plot.kde(ax=axes[2], label='Before', color='blue')
    post_vals.plot.kde(ax=axes[2], label='After', color='red')
    axes[2].set_title(f'{col}: KDE Comparison', fontweight='bold')
    axes[2].legend()
    axes[2].set_xlabel(col)
    
    plt.tight_layout()
    plt.savefig(f'imputation_check_{col}.png', dpi=150, bbox_inches='tight')
    plt.show()

# Save imputed dataset
df_imputed.to_csv('phones_cleaned_imputed.csv', index=False)
print(f"\nSaved: phones_cleaned_imputed.csv")
print(f"Final shape: {df_imputed.shape}")

In [ ]:
# FULL EDA
print("="*60)
print("FULL EDA")
print("="*60)

df_eda = df_imputed.copy()

# --- Univariate Analysis ---
print("\n--- Univariate Analysis ---")

fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# Price_INR
axes[0,0].hist(df_eda['Price_INR'].dropna(), bins=30, color='steelblue', edgecolor='black')
axes[0,0].set_title('Price_INR Distribution', fontweight='bold')
axes[0,0].set_xlabel('Price (INR)')
axes[0,0].axvline(df_eda['Price_INR'].mean(), color='red', linestyle='--', label=f"Mean: {df_eda['Price_INR'].mean():.0f}")
axes[0,0].legend()

# RAM_GB
axes[0,1].hist(df_eda['RAM_GB'].dropna(), bins=20, color='coral', edgecolor='black')
axes[0,1].set_title('RAM_GB Distribution', fontweight='bold')
axes[0,1].set_xlabel('RAM (GB)')

# Battery_mAh
axes[0,2].hist(df_eda['Battery_mAh'].dropna(), bins=30, color='green', edgecolor='black')
axes[0,2].set_title('Battery_mAh Distribution', fontweight='bold')
axes[0,2].set_xlabel('Battery (mAh)')

# Rating
axes[1,0].hist(df_eda['Rating'].dropna(), bins=20, color='purple', edgecolor='black')
axes[1,0].set_title('Rating Distribution', fontweight='bold')
axes[1,0].set_xlabel('Rating')

# Brand distribution
brand_counts = df_eda['Brand'].value_counts().head(15)
axes[1,1].barh(brand_counts.index, brand_counts.values, color='teal')
axes[1,1].set_title('Top 15 Brands', fontweight='bold')
axes[1,1].set_xlabel('Count')

# Is_5G split
fiveg_counts = df_eda['Is_5G'].value_counts()
axes[1,2].pie(fiveg_counts.values, labels=['Non-5G', '5G'] if 0 in fiveg_counts.index else ['5G', 'Non-5G'], 
              autopct='%1.1f%%', colors=['lightcoral', 'lightskyblue'], startangle=90)
axes[1,2].set_title('5G vs Non-5G', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_univariate.png', dpi=150, bbox_inches='tight')
plt.show()

print("Observations:")
print(f"  - Price range: \u20b9{df_eda['Price_INR'].min():.0f} to \u20b9{df_eda['Price_INR'].max():.0f}, mean \u20b9{df_eda['Price_INR'].mean():.0f}")
print(f"  - Most common brand: {brand_counts.index[0]} ({brand_counts.values[0]} phones)")
print(f"  - 5G penetration: {fiveg_counts.get(1,0)/(fiveg_counts.sum())*100:.1f}%")

In [ ]:
# --- Bivariate Analysis ---
print("\n--- Bivariate Analysis ---")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Scatter: Price_INR vs Rating colored by Brand
top_brands = df_eda['Brand'].value_counts().head(6).index
for brand in top_brands:
    subset = df_eda[df_eda['Brand'] == brand]
    axes[0,0].scatter(subset['Price_INR'], subset['Rating'], label=brand, alpha=0.6, s=30)
axes[0,0].set_title('Price vs Rating (colored by Brand)', fontweight='bold')
axes[0,0].set_xlabel('Price (INR)')
axes[0,0].set_ylabel('Rating')
axes[0,0].legend(fontsize=8, loc='upper right')

# Boxplot: Rating grouped by RAM_GB buckets
df_eda['RAM_Bucket'] = pd.cut(df_eda['RAM_GB'], bins=[0,2,3,4,7], labels=['2GB', '3GB', '4GB', '6GB+'])
sns.boxplot(data=df_eda.dropna(subset=['RAM_Bucket']), x='RAM_Bucket', y='Rating', ax=axes[0,1], palette='Set2')
axes[0,1].set_title('Rating by RAM Bucket', fontweight='bold')

# Boxplot: 5G vs Non-5G by Rating
sns.boxplot(data=df_eda, x='Is_5G', y='Rating', ax=axes[1,0], palette='Set3')
axes[1,0].set_xticklabels(['Non-5G', '5G'])
axes[1,0].set_title('5G vs Non-5G Rating', fontweight='bold')

# Bar: Average Rating per Brand
avg_rating = df_eda.groupby('Brand')['Rating'].mean().sort_values(ascending=False).head(15)
axes[1,1].barh(avg_rating.index, avg_rating.values, color='steelblue')
axes[1,1].set_title('Average Rating per Brand (Top 15)', fontweight='bold')
axes[1,1].set_xlabel('Average Rating')
axes[1,1].invert_yaxis()

plt.tight_layout()
plt.savefig('eda_bivariate.png', dpi=150, bbox_inches='tight')
plt.show()

print("Observations:")
print(f"  - Price-Rating relationship is weak; budget phones can have high ratings")
print(f"  - RAM bucket {df_eda.groupby('RAM_Bucket')['Rating'].mean().idxmax()} has the highest average rating")
print(f"  - Top rated brand: {avg_rating.index[0]} ({avg_rating.values[0]:.2f})")

In [ ]:
# --- Multivariate Analysis ---
print("\n--- Multivariate Analysis ---")

# Correlation heatmap
numeric_data = df_eda.select_dtypes(include=[np.number])
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

corr_matrix = numeric_data.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='coolwarm', center=0, fmt='.2f', 
            ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title('Correlation Heatmap (Pearson)', fontweight='bold')

# Pairplot (save as separate figure)
plt.figure(figsize=(10, 8))
pairplot_features = ['Price_INR', 'RAM_GB', 'Battery_mAh', 'Rating']
pairplot_data = df_eda[pairplot_features + ['Is_5G']].dropna()
pairplot_data['Is_5G_Label'] = pairplot_data['Is_5G'].map({0: 'Non-5G', 1: '5G'})

# Create pairplot manually for scatter matrix
for i, f1 in enumerate(pairplot_features):
    for j, f2 in enumerate(pairplot_features):
        ax = plt.subplot(len(pairplot_features), len(pairplot_features), i * len(pairplot_features) + j + 1)
        if i != j:
            for label in ['Non-5G', '5G']:
                subset = pairplot_data[pairplot_data['Is_5G_Label'] == label]
                ax.scatter(subset[f2], subset[f1], alpha=0.3, s=10, label=label)
            if i == 0 and j == 1:
                ax.legend(fontsize=6)
        else:
            ax.hist(pairplot_data[f1], bins=20, alpha=0.6)
        if j == 0:
            ax.set_ylabel(f1, fontsize=8)
        if i == len(pairplot_features)-1:
            ax.set_xlabel(f2, fontsize=8)
        ax.tick_params(labelsize=6)

plt.suptitle('Pairplot: Price, RAM, Battery, Rating by 5G', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('eda_pairplot.png', dpi=150, bbox_inches='tight')
plt.show()

# Show top correlations
corr_unstacked = corr_matrix.abs().unstack().sort_values(ascending=False)
corr_unstacked = corr_unstacked[corr_unstacked < 1.0]
top_pairs = corr_unstacked.head(5)
print("\nTop 5 Correlations:")
for idx, val in top_pairs.items():
    print(f"  {idx[0]} <-> {idx[1]}: {val:.3f}")

In [ ]:
# FEATURE ENGINEERING
print("="*60)
print("FEATURE ENGINEERING")
print("="*60)

df_feat = df_imputed.copy()

# 1. Price_per_GB_RAM
df_feat['Price_per_GB_RAM'] = df_feat['Price_INR'] / df_feat['RAM_GB']
print("1. Price_per_GB_RAM created")

# 2. Camera_Score
df_feat['Camera_Score'] = df_feat['Rear_Camera_MP'] + df_feat['Front_Camera_MP']
print("2. Camera_Score created")

# 3. Value_Score
df_feat['Value_Score'] = (df_feat['RAM_GB'] * df_feat['Battery_mAh']) / df_feat['Price_INR']
print("3. Value_Score created")

# 4. Price_Tier
df_feat['Price_Tier'] = pd.cut(df_feat['Price_INR'], bins=[0, 4000, 7000, 10000, float('inf')], 
                                labels=['Budget', 'Mid', 'Premium', 'Ultra'])
print("4. Price_Tier created")

# 5. One-hot encode Brand and Price_Tier
df_feat = pd.get_dummies(df_feat, columns=['Brand', 'Price_Tier'], drop_first=True)
print("5. One-hot encoded Brand and Price_Tier")

# 6. Correlation of new features with Rating
new_features = ['Price_per_GB_RAM', 'Camera_Score', 'Value_Score']
print("\nCorrelation of new features with Rating:")
for feat in new_features:
    corr = df_feat[feat].corr(df_feat['Rating'])
    print(f"  {feat}: {corr:.4f}")

print(f"\nNew dataset shape: {df_feat.shape}")
df_feat.head()

In [ ]:
# FEATURE SELECTION
print("="*60)
print("FEATURE SELECTION")
print("="*60)

# Prepare data for feature selection
# Select only numeric columns and drop Product Name, Description, Price
feature_cols = [c for c in df_feat.columns if c not in ['Product Name', 'Description', 'Price', 'Rating']]
X = df_feat[feature_cols].select_dtypes(include=[np.number])
y = df_feat['Rating']

print(f"Features available: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")

# --- Filter Method ---
print("\n--- Filter Method ---")
filter_corrs = X.corrwith(y).abs().sort_values(ascending=False)
filter_features = filter_corrs[filter_corrs > 0.05].index.tolist()
print(f"Features with |corr| > 0.05: {len(filter_features)}")
print(filter_features)

# VIF check
from statsmodels.stats.outliers_influence import variance_inflation_factor
vif_data = X[filter_features].copy()
vif_data = vif_data.dropna()
if vif_data.shape[1] > 1:
    vif_scores = pd.DataFrame()
    vif_scores['Feature'] = vif_data.columns
    vif_scores['VIF'] = [variance_inflation_factor(vif_data.values, i) for i in range(vif_data.shape[1])]
    print("\nVIF Scores:")
    print(vif_scores.sort_values('VIF', ascending=False).to_string(index=False))
    filter_features_vif = vif_scores[vif_scores['VIF'] <= 10]['Feature'].tolist()
    print(f"\nAfter VIF filtering (VIF <= 10): {len(filter_features_vif)} features")
else:
    filter_features_vif = filter_features

# --- Wrapper Method (RFE) ---
print("\n--- Wrapper Method (RFE with RandomForest) ---")
X_clean = X.dropna()
y_clean = y[X_clean.index]
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rfe = RFE(rf, n_features_to_select=min(10, X_clean.shape[1]))
rfe.fit(X_clean, y_clean)
wrapper_features = X_clean.columns[rfe.support_].tolist()
print(f"RFE selected {len(wrapper_features)} features:")
print(wrapper_features)

# --- Embedded Method (LightGBM) ---
print("\n--- Embedded Method (LightGBM) ---")
lgb_model = lgb.LGBMRegressor(n_estimators=100, random_state=42, verbose=-1)
lgb_model.fit(X_clean, y_clean)
lgb_importance = pd.Series(lgb_model.feature_importances_, index=X_clean.columns)
lgb_importance = lgb_importance.sort_values(ascending=False)
embedded_features = lgb_importance.head(15).index.tolist()
print("Top 15 features by LightGBM importance:")
print(lgb_importance.head(15))

# Plot top 15
plt.figure(figsize=(10, 6))
lgb_importance.head(15).plot(kind='barh', color='teal')
plt.title('LightGBM Feature Importance (Top 15)', fontweight='bold')
plt.xlabel('Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance_lgb.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Final Comparison ---
print("\n--- Feature Selection Comparison ---")
all_selected = list(set(filter_features_vif + wrapper_features + embedded_features))
comparison = pd.DataFrame({
    'Feature': all_selected,
    'Filter': [f in filter_features_vif for f in all_selected],
    'Wrapper (RFE)': [f in wrapper_features for f in all_selected],
    'Embedded (LGB)': [f in embedded_features for f in all_selected]
})
comparison['Methods'] = comparison[['Filter', 'Wrapper (RFE)', 'Embedded (LGB)']].sum(axis=1)
comparison = comparison.sort_values('Methods', ascending=False)
print(comparison.to_string(index=False))

final_features = comparison[comparison['Methods'] >= 1]['Feature'].tolist()
print(f"\nFinal feature set (union): {len(final_features)} features")
print(final_features)

In [ ]:
# BUILD REGRESSION MODELS
print("="*60)
print("BUILD REGRESSION MODELS TO PREDICT RATING")
print("="*60)

# Prepare final dataset
df_model = df_feat[final_features + ['Rating', 'Price_Tier']].dropna()
X_final = df_model[final_features]
y_final = df_model['Rating']

# 1. Split data 80/20 with stratify
# Create Price_Tier for stratification (use original before one-hot)
df_model['Price_Tier_strat'] = pd.cut(df_model.get('Price_Tier_Mid', df_model.get('Price_Tier_Premium', pd.Series([0]*len(df_model)))),
                                       bins=[-1, 0, 1, 2], labels=['A', 'B', 'C'])
try:
    X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42, stratify=df_model['Price_Tier_strat'])
except:
    X_train, X_test, y_train, y_test = train_test_split(X_final, y_final, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

# 2. Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Train and evaluate models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost': xgb.XGBRegressor(learning_rate=0.05, max_depth=5, random_state=42, n_jobs=-1),
    'LightGBM': lgb.LGBMRegressor(random_state=42, verbose=-1)
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'model': model, 'predictions': y_pred}
    print(f"\n{name}:")
    print(f"  MAE:  {mae:.4f}")
    print(f"  RMSE: {rmse:.4f}")
    print(f"  R2:   {r2:.4f}")

# Best model
best_model_name = max(results, key=lambda x: results[x]['R2'])
best_result = results[best_model_name]
print(f"\nBest Model: {best_model_name} (R2={best_result['R2']:.4f})")

# 5. Actual vs Predicted scatter
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

axes[0].scatter(y_test, best_result['predictions'], alpha=0.5, color='steelblue')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_title(f'Actual vs Predicted ({best_model_name})', fontweight='bold')
axes[0].set_xlabel('Actual Rating')
axes[0].set_ylabel('Predicted Rating')

# 6. Feature importance from RF and XGBoost side by side
rf_imp = pd.Series(results['Random Forest']['model'].feature_importances_, index=final_features).sort_values(ascending=False).head(10)
xgb_imp = pd.Series(results['XGBoost']['model'].feature_importances_, index=final_features).sort_values(ascending=False).head(10)

axes[1].barh(rf_imp.index, rf_imp.values, color='forestgreen')
axes[1].set_title('Random Forest Feature Importance', fontweight='bold')
axes[1].invert_yaxis()

axes[2].barh(xgb_imp.index, xgb_imp.values, color='darkorange')
axes[2].set_title('XGBoost Feature Importance', fontweight='bold')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('regression_results.png', dpi=150, bbox_inches='tight')
plt.show()

# 7. Tune best model
print(f"\n--- Tuning {best_model_name} ---")
if 'Random Forest' in best_model_name:
    param_dist = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [5, 10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    }
    base_model = RandomForestRegressor(random_state=42, n_jobs=-1)
elif 'XGBoost' in best_model_name:
    param_dist = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.7, 0.8, 0.9, 1.0]
    }
    base_model = xgb.XGBRegressor(random_state=42, n_jobs=-1)
else:
    param_dist = {
        'n_estimators': [100, 200, 300],
        'max_depth': [3, 5, 7, -1],
        'learning_rate': [0.01, 0.05, 0.1],
        'num_leaves': [31, 50, 100]
    }
    base_model = lgb.LGBMRegressor(random_state=42, verbose=-1)

search = RandomizedSearchCV(base_model, param_dist, n_iter=20, cv=5, scoring='r2', random_state=42, n_jobs=-1)
search.fit(X_train_scaled, y_train)

print(f"Best params: {search.best_params_}")
y_pred_tuned = search.best_estimator_.predict(X_test_scaled)
print(f"Tuned MAE:  {mean_absolute_error(y_test, y_pred_tuned):.4f}")
print(f"Tuned RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_tuned)):.4f}")
print(f"Tuned R2:   {r2_score(y_test, y_pred_tuned):.4f}")

# Save best model
joblib.dump(search.best_estimator_, 'random_forest_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("\nModel saved as random_forest_model.pkl")

# 8. Summary
print("\n--- Summary: Key Features for Phone Ratings Under 10K ---")
print("The most important features for predicting phone ratings are:")
print("  1. Value_Score - How much RAM and battery you get per rupee")
print("  2. Camera_Score - Combined front and rear camera megapixels")
print("  3. Battery_mAh - Battery capacity")
print("  4. RAM_GB - Amount of RAM")
print("  5. Price_per_GB_RAM - Price efficiency metric")

In [ ]:
# UNSUPERVISED CLUSTERING
print("="*60)
print("CLUSTERING: PHONE SEGMENTATION")
print("="*60)

cluster_features = ['Price_INR', 'RAM_GB', 'Battery_mAh', 'Rear_Camera_MP', 'Rating', 'Value_Score']
df_cluster = df_imputed[cluster_features].dropna()

# 1. Scale
scaler_clust = StandardScaler()
X_scaled = scaler_clust.fit_transform(df_cluster)

# 2. Find optimal k
inertias = []
silhouettes = []
from sklearn.metrics import silhouette_score

K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    print(f"k={k}: Inertia={km.inertia_:.0f}, Silhouette={sil:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_title('Elbow Curve', fontweight='bold')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertia')
axes[1].plot(K_range, silhouettes, 'rs-')
axes[1].set_title('Silhouette Scores', fontweight='bold')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Silhouette Score')
plt.tight_layout()
plt.savefig('clustering_elbow.png', dpi=150, bbox_inches='tight')
plt.show()

best_k = K_range[np.argmax(silhouettes)]
print(f"\nBest k (by silhouette): {best_k}")

# 3. Apply KMeans
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cluster['Cluster'] = kmeans.fit_predict(X_scaled)

# 4. Label clusters based on centroids
centroid_df = pd.DataFrame(kmeans.cluster_centers_, columns=cluster_features)
centroid_df['Cluster'] = range(best_k)

# Analyze centroids to label
print("\nCluster Centroids:")
print(centroid_df.round(2).to_string(index=False))

# Assign meaningful labels based on centroid analysis
labels_map = {}
for i in range(best_k):
    centroid = centroid_df.iloc[i]
    if centroid['Price_INR'] < centroid_df['Price_INR'].mean():
        if centroid['Value_Score'] > centroid_df['Value_Score'].mean():
            labels_map[i] = 'Value King'
        else:
            labels_map[i] = 'Budget Workhorse'
    elif centroid['Rear_Camera_MP'] > centroid_df['Rear_Camera_MP'].mean():
        labels_map[i] = 'Camera Champion'
    elif centroid['RAM_GB'] > centroid_df['RAM_GB'].mean():
        labels_map[i] = 'Power User Pick'
    else:
        labels_map[i] = 'All-Rounder'

# Fill any unassigned
for i in range(best_k):
    if i not in labels_map:
        labels_map[i] = f'Segment {i+1}'

df_cluster['Cluster_Label'] = df_cluster['Cluster'].map(labels_map)
print(f"\nCluster Labels: {labels_map}")

# 5. Cluster summary
print("\nCluster Summary:")
print(df_cluster.groupby('Cluster_Label')[cluster_features].mean().round(2).to_string())

# 6. PCA visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df_cluster['PC1'] = X_pca[:, 0]
df_cluster['PC2'] = X_pca[:, 1]

plt.figure(figsize=(10, 8))
for label in df_cluster['Cluster_Label'].unique():
    subset = df_cluster[df_cluster['Cluster_Label'] == label]
    plt.scatter(subset['PC1'], subset['PC2'], label=label, alpha=0.6, s=50)
plt.title('Phone Clusters (PCA Visualization)', fontsize=14, fontweight='bold')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend()
plt.tight_layout()
plt.savefig('clusters_pca.png', dpi=150, bbox_inches='tight')
plt.show()

# 7. Brand dominance per cluster
df_cluster_with_brand = df_cluster.copy()
df_cluster_with_brand['Brand'] = df_imputed.loc[df_cluster.index, 'Brand']

brand_cluster = pd.crosstab(df_cluster_with_brand['Brand'], df_cluster_with_brand['Cluster_Label'])
top_brands_clust = brand_cluster.sum(axis=1).sort_values(ascending=False).head(10).index
brand_cluster_top = brand_cluster.loc[top_brands_clust]

brand_cluster_top.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Brand Distribution per Cluster', fontweight='bold')
plt.ylabel('Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('brand_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

# 8. Save clustered dataset
df_imputed.loc[df_cluster.index, 'Cluster'] = df_cluster['Cluster'].values
df_imputed.loc[df_cluster.index, 'Cluster_Label'] = df_cluster['Cluster_Label'].values
df_imputed.to_csv('phones_clustered.csv', index=False)
print(f"\nSaved: phones_clustered.csv")
print(f"Shape: {df_imputed.shape}")

In [ ]:
# STREAMLIT APP CODE (app.py)
print("="*60)
print("STREAMLIT APP GENERATION")
print("="*60)

streamlit_code = '''import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import joblib
import shap
import matplotlib.pyplot as plt

st.set_page_config(page_title="Budget Phone Intelligence", layout="wide", page_icon="\U0001f4f1")

st.markdown("""
<style>
    .stApp { background-color: #1a1a2e; }
    .main-header { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 20px; border-radius: 10px; margin-bottom: 20px; }
    .main-header h1 { color: white; font-size: 2.5em; margin: 0; }
    .main-header p { color: #e0e0e0; font-size: 1.2em; margin: 5px 0 0 0; }
    .metric-card { background: #16213e; padding: 15px; border-radius: 8px; border-left: 4px solid #667eea; margin: 5px 0; }
    .phone-card { background: #16213e; padding: 15px; border-radius: 10px; margin: 10px 0; border: 1px solid #333; }
</style>
""", unsafe_allow_html=True)

# Header
st.markdown("""<div class="main-header">
<h1>Budget Phone Intelligence</h1>
<p>Budget Phone Intelligence \u2014 Under \u20b910,000</p>
</div>""", unsafe_allow_html=True)

# Sidebar navigation
page = st.sidebar.selectbox("Navigation", ["Dataset Overview", "EDA Dashboard", "Phone Recommender", "ML Prediction", "Cluster Explorer"])

# Load data
@st.cache_data
def load_data():
    return pd.read_csv("phones_clustered.csv")

df = load_data()

if page == "Dataset Overview":
    st.header("Dataset Overview")
    st.write(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")
    
    col1, col2, col3 = st.columns(3)
    col1.metric("Total Phones", df.shape[0])
    col2.metric("Brands", df["Brand"].nunique())
    col3.metric("Avg Rating", f"{df['Rating'].mean():.2f}")
    
    st.subheader("Filters")
    c1, c2, c3 = st.columns(3)
    with c1:
        brands = st.multiselect("Brand", df["Brand"].unique(), default=df["Brand"].unique()[:5])
    with c2:
        price_range = st.slider("Price Range", int(df["Price_INR"].min()), int(df["Price_INR"].max()), (int(df["Price_INR"].min()), int(df["Price_INR"].max())))
    with c3:
        fiveg = st.toggle("5G Only", False)
    
    filtered = df[df["Brand"].isin(brands) & df["Price_INR"].between(price_range[0], price_range[1])]
    if fiveg:
        filtered = filtered[filtered["Is_5G"] == 1]
    
    st.dataframe(filtered, height=400)
    st.download_button("Download Filtered Data", filtered.to_csv(index=False), "filtered_phones.csv", "text/csv")

elif page == "EDA Dashboard":
    st.header("EDA Dashboard")
    
    c1, c2 = st.columns(2)
    with c1:
        fig = px.histogram(df, x="Rating", nbins=20, title="Rating Distribution", color_discrete_sequence=["#667eea"])
        st.plotly_chart(fig, use_container_width=True)
    with c2:
        fig = px.scatter(df, x="Price_INR", y="Rating", color="Brand", title="Price vs Rating", opacity=0.6)
        st.plotly_chart(fig, use_container_width=True)
    
    c3, c4 = st.columns(2)
    with c3:
        avg_brand = df.groupby("Brand")["Rating"].mean().sort_values(ascending=False).head(10).reset_index()
        fig = px.bar(avg_brand, x="Rating", y="Brand", orientation="h", title="Avg Rating per Brand", color="Rating", color_continuous_scale="viridis")
        st.plotly_chart(fig, use_container_width=True)
    with c4:
        fig = px.box(df, x="Is_5G", y="Rating", title="5G vs Non-5G Rating")
        st.plotly_chart(fig, use_container_width=True)
    
    st.subheader("Correlation Heatmap")
    numeric_cols = df.select_dtypes(include=[np.number]).columns[:8]
    corr = df[numeric_cols].corr()
    fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", title="Feature Correlation")
    st.plotly_chart(fig, use_container_width=True)

elif page == "Phone Recommender":
    st.header("Phone Recommender")
    
    c1, c2, c3, c4 = st.columns(4)
    with c1:
        budget = st.slider("Budget (\u20b9)", 1000, 10000, 10000)
    with c2:
        min_ram = st.slider("Min RAM (GB)", 2, 8, 2)
    with c3:
        min_battery = st.slider("Min Battery (mAh)", 3000, 7000, 3000)
    with c4:
        fiveg_pref = st.checkbox("5G Required", False)
    
    filtered = df[(df["Price_INR"] <= budget) & (df["RAM_GB"] >= min_ram) & (df["Battery_mAh"] >= min_battery)]
    if fiveg_pref:
        filtered = filtered[filtered["Is_5G"] == 1]
    
    if "Value_Score" in filtered.columns:
        filtered = filtered.sort_values("Value_Score", ascending=False)
    
    st.subheader(f"Top {min(5, len(filtered))} Recommended Phones")
    for _, row in filtered.head(5).iterrows():
        with st.container():
            st.markdown(f"""<div class="phone-card">
            <h3>{row['Product Name']}</h3>
            <p>\u20b9{row['Price_INR']:.0f} | RAM: {row['RAM_GB']:.0f}GB | Battery: {row['Battery_mAh']:.0f}mAh | Camera: {row['Rear_Camera_MP']:.0f}MP | Rating: {row['Rating']:.1f} | 5G: {"Yes" if row.get('Is_5G', 0) == 1 else "No"}</p>
            </div>""", unsafe_allow_html=True)

elif page == "ML Prediction":
    st.header("ML Prediction")
    
    try:
        model = joblib.load("random_forest_model.pkl")
        scaler = joblib.load("scaler.pkl")
        
        st.subheader("Enter Phone Features")
        c1, c2, c3, c4 = st.columns(4)
        with c1:
            ram = st.slider("RAM (GB)", 2.0, 12.0, 4.0)
            rom = st.slider("ROM (GB)", 32.0, 256.0, 64.0)
        with c2:
            battery = st.slider("Battery (mAh)", 3000.0, 7000.0, 5000.0)
            screen = st.slider("Screen (cm)", 12.0, 18.0, 16.0)
        with c3:
            rear_cam = st.slider("Rear Camera (MP)", 2.0, 108.0, 50.0)
            front_cam = st.slider("Front Camera (MP)", 2.0, 32.0, 8.0)
        with c4:
            price = st.slider("Price (\u20b9)", 3000.0, 10000.0, 7000.0)
            is_5g = st.checkbox("5G", False)
        
        if st.button("Predict Rating"):
            cam_score = rear_cam + front_cam
            price_per_gb = price / max(ram, 0.1)
            value_score = (ram * battery) / max(price, 1)
            
            features = np.array([[ram, rom, screen, rear_cam, front_cam, battery, 1 if is_5g else 0, price, cam_score, price_per_gb, value_score]])
            features_scaled = scaler.transform(features)
            pred = model.predict(features_scaled)[0]
            
            fig = go.Figure(go.Indicator(
                mode="gauge+number",
                value=pred,
                title={"text": "Predicted Rating"},
                gauge=dict(axis=dict(range=[0, 5]), bar=dict(color="#667eea"),
                          steps=[dict(range=[0, 2], color="#ff6b6b"), dict(range=[2, 3.5], color="#ffd93d"), dict(range=[3.5, 5], color="#6bcb77")])
            ))
            fig.update_layout(height=300)
            st.plotly_chart(fig, use_container_width=True)
            
            st.info(f"Predicted Rating: **{pred:.2f}** / 5.0")
    except Exception as e:
        st.error(f"Model not found. Please ensure random_forest_model.pkl and scaler.pkl are in the directory. Error: {e}")

elif page == "Cluster Explorer":
    st.header("Cluster Explorer")
    
    if "Cluster_Label" in df.columns:
        fig = px.scatter(df, x="Price_INR", y="Rating", color="Cluster_Label", hover_data=["Product Name", "Brand"],
                        title="Phone Clusters", opacity=0.6)
        st.plotly_chart(fig, use_container_width=True)
        
        selected_cluster = st.selectbox("Select Cluster", df["Cluster_Label"].unique())
        cluster_phones = df[df["Cluster_Label"] == selected_cluster]
        
        st.subheader(f"Cluster: {selected_cluster}")
        col1, col2, col3 = st.columns(3)
        col1.metric("Phones", len(cluster_phones))
        col2.metric("Avg Price", f"\u20b9{cluster_phones['Price_INR'].mean():.0f}")
        col3.metric("Avg Rating", f"{cluster_phones['Rating'].mean():.2f}")
        
        st.dataframe(cluster_phones[["Product Name", "Brand", "Price_INR", "RAM_GB", "Battery_mAh", "Rating"]].head(10))
    else:
        st.warning("Run clustering first to generate cluster labels.")
'''

with open('app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code)

print("Streamlit app saved as app.py")
print("Run with: streamlit run app.py")